# Lab Notebook 2: Cropland Classification
**Team:** MozwiloFortune

## Purpose
Train a CNN to classify cropland versus non-cropland using the FIXED dataset with global normalization.

## Prerequisites
- Run Lab Notebook 1 first to create the dataset
- Dataset should be in `/p/scratch/training2600/teamMozwiloFortune/final_project/data/`

## Expected Results
- Balanced Accuracy: 70-85%
- Macro-F1: 70-85%

## Runtime: ~1-2 hours (GPU) or ~3-4 hours (CPU)

## Setup

In [1]:
import os
from pathlib import Path
import numpy as np
import json
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.metrics import (
    balanced_accuracy_score, f1_score, classification_report,
    precision_recall_fscore_support, confusion_matrix
)

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("✓ Libraries imported")
print(f"  PyTorch: {torch.__version__}")
print(f"  Lightning: {pl.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

✓ Libraries imported
  PyTorch: 2.10.0+cu128
  Lightning: 2.6.1
  CUDA available: True


## Configuration

In [2]:
# Paths
DATA_DIR = Path("/p/scratch/training2600/teamMozwiloFortune/final_project/data")
OUTPUT_DIR = Path("/p/scratch/training2600/teamMozwiloFortune/final_project")

# Training
BATCH_SIZE = 128
MAX_EPOCHS = 50
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.0001
PATIENCE = 10
NUM_WORKERS = 4
CLASS_WEIGHT_POWER = 0.5
LOSS_NAME = 'cross_entropy'
DROPOUT = 0.3

# Manual tuning setup
BEST_CONFIG = {
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'dropout': DROPOUT,
    'class_weight_power': CLASS_WEIGHT_POWER
}
TUNING_CANDIDATES = [
    {'name': 'baseline', 'batch_size': 128, 'learning_rate': 3e-4, 'weight_decay': 1e-4, 'dropout': 0.3, 'class_weight_power': 0.5},
    {'name': 'smaller_batch', 'batch_size': 64, 'learning_rate': 3e-4, 'weight_decay': 1e-4, 'dropout': 0.3, 'class_weight_power': 0.5},
    {'name': 'lower_lr', 'batch_size': 128, 'learning_rate': 1e-4, 'weight_decay': 1e-4, 'dropout': 0.3, 'class_weight_power': 0.5},
    {'name': 'more_dropout', 'batch_size': 128, 'learning_rate': 3e-4, 'weight_decay': 1e-4, 'dropout': 0.5, 'class_weight_power': 0.5},
    {'name': 'weaker_class_weights', 'batch_size': 128, 'learning_rate': 3e-4, 'weight_decay': 1e-4, 'dropout': 0.3, 'class_weight_power': 0.0}
]

# Create output dirs
(OUTPUT_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"  Data: {DATA_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Loss: {LOSS_NAME}")
print(f"  Current config: {BEST_CONFIG}")
print(f"  Manual tuning candidates: {len(TUNING_CANDIDATES)}")
for candidate in TUNING_CANDIDATES:
    print(f"    - {candidate['name']}: {candidate}")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Configuration:
  Data: /p/scratch/training2600/teamMozwiloFortune/final_project/data
  Output: /p/scratch/training2600/teamMozwiloFortune/final_project
  Batch size: 256
  Max epochs: 50
  Device: cuda


## Load Dataset

In [3]:
print("="*70)
print("LOADING DATASET")
print("="*70)

# Load data
data = {}
for split in ['train', 'val', 'test']:
    patches_file = np.load(DATA_DIR / f'patches_{split}.npz')
    patches = patches_file['patches']
    labels = np.load(DATA_DIR / f'labels_{split}.npy')
    data[split] = {'patches': patches, 'labels': labels}
    print(f"{split:5s}: {len(labels):6,} patches, shape {patches.shape}")

# Load metadata
with open(DATA_DIR / 'metadata.json') as f:
    metadata = json.load(f)

print(f"\nNormalization: {metadata['normalization']}")
print(f"Bands: {metadata['n_bands_total']}")
print(f"\n✅ Dataset loaded")

LOADING DATASET
train: 20,000 patches, shape (20000, 3, 3, 16)
val  : 10,000 patches, shape (10000, 3, 3, 16)
test : 10,000 patches, shape (10000, 3, 3, 16)

Normalization: GLOBAL_percentile_2_98
Bands: 16

✅ Dataset loaded


## Remap Labels

In [4]:
print("\n" + "="*70)
print("BINARY LABEL REMAPPING: CROPLAND VS NON-CROPLAND")
print("="*70)

# CORINE labels treated as cropland in this project
cropland_labels = {12, 15, 20, 21}

# Inspect original label space
all_labels = np.concatenate([
    data['train']['labels'],
    data['val']['labels'],
    data['test']['labels']
])
unique_labels = np.unique(all_labels)

print(f"\nOriginal CORINE labels in dataset: {sorted(unique_labels.tolist())}")
print(f"Cropland labels: {sorted(cropland_labels)}")
print(f"Non-cropland labels: {sorted([int(lbl) for lbl in unique_labels if int(lbl) not in cropland_labels])}")

# Binary remap: 1 = cropland, 0 = non-cropland
NUM_CLASSES = 2
idx_to_label = {0: 'non_cropland', 1: 'cropland'}
label_to_idx = {'non_cropland': 0, 'cropland': 1}

data_remapped = {}
for split in ['train', 'val', 'test']:
    patches = data[split]['patches'].astype(np.float32)
    labels = np.array([
        1 if int(lbl) in cropland_labels else 0
        for lbl in data[split]['labels']
    ], dtype=np.int64)
    data_remapped[split] = {'patches': patches, 'labels': labels}

    counts = np.bincount(labels, minlength=NUM_CLASSES)
    print(f"{split:5s}: non-cropland={counts[0]:6,}, cropland={counts[1]:6,}")

print(f"\nNum classes: {NUM_CLASSES}")
print("✅ Binary labels remapped")


LABEL REMAPPING

CORINE labels: [1, 2, 3, 12, 15, 18, 20, 21, 23, 24, 25, 26, 27, 29, 30, 31, 32, 34, 41]
Mapped to: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

Num classes: 19
✅ Labels remapped


## Create DataLoaders

In [5]:
class PatchDataset(Dataset):
    def __init__(self, patches, labels):
        self.patches = torch.from_numpy(patches).float()
        self.labels = torch.from_numpy(labels).long()
        # (N, H, W, C) → (N, C, H, W)
        self.patches = self.patches.permute(0, 3, 1, 2)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.patches[idx], self.labels[idx]

# Create datasets
train_dataset = PatchDataset(data_remapped['train']['patches'], data_remapped['train']['labels'])
val_dataset = PatchDataset(data_remapped['val']['patches'], data_remapped['val']['labels'])
test_dataset = PatchDataset(data_remapped['test']['patches'], data_remapped['test']['labels'])

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print("\n" + "="*70)
print("DATALOADERS CREATED")
print("="*70)
print(f"Train: {len(train_loader):3d} batches × {BATCH_SIZE} = {len(train_dataset):,}")
print(f"Val:   {len(val_loader):3d} batches × {BATCH_SIZE} = {len(val_dataset):,}")
print(f"Test:  {len(test_loader):3d} batches × {BATCH_SIZE} = {len(test_dataset):,}")


DATALOADERS CREATED
Train:  79 batches × 256 = 20,000
Val:    40 batches × 256 = 10,000
Test:   40 batches × 256 = 10,000


## Define Model

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels=16, num_classes=25, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

def build_class_weights(train_labels, num_classes, class_weight_power):
    class_counts = np.bincount(train_labels, minlength=num_classes)
    weights = np.ones(num_classes, dtype=np.float32)
    present_mask = class_counts > 0
    weights[present_mask] = 1.0 / np.power(class_counts[present_mask].astype(np.float32), class_weight_power)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

class LandCoverClassifier(pl.LightningModule):
    def __init__(self, model, num_classes, class_weights, learning_rate=0.001, weight_decay=0.0001, loss_name='cross_entropy'):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.loss_name = loss_name
        self.register_buffer('class_weights', class_weights)
        
        self.save_hyperparameters(ignore=['model'])
    
    def forward(self, x):
        return self.model(x)

    def compute_loss(self, logits, y):
        return F.cross_entropy(logits, y, weight=self.class_weights)
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.compute_loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_acc', acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.compute_loss(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate, 
                                     weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val_loss'}
        }

# Initialize baseline model for reference
baseline_class_weights = build_class_weights(
    data_remapped['train']['labels'],
    NUM_CLASSES,
    CLASS_WEIGHT_POWER
)
cnn_model = SimpleCNN(in_channels=16, num_classes=NUM_CLASSES, dropout=0.3)
lit_model = LandCoverClassifier(
    cnn_model,
    NUM_CLASSES,
    baseline_class_weights,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    loss_name=LOSS_NAME
)

total_params = sum(p.numel() for p in cnn_model.parameters())

print("\n" + "="*70)
print("MODEL")
print("="*70)
print(f"Architecture: SimpleCNN")
print(f"Input: 16 channels (4 bands × 4 dates)")
print(f"Output: {NUM_CLASSES} classes")
print(f"Loss: {LOSS_NAME}")
print(f"Parameters: {total_params:,}")


MODEL
Architecture: SimpleCNN
Input: 16 channels (4 bands × 4 dates)
Output: 19 classes
Parameters: 384,083


## Train Model

In [9]:
def make_loaders(train_patches, train_labels, val_patches, val_labels, batch_size):
    train_ds = PatchDataset(train_patches, train_labels)
    val_ds = PatchDataset(val_patches, val_labels)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    return train_dl, val_dl

def predict_with_model(model, dataloader, device):
    model.eval()
    model = model.to(device)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.numpy())
    return np.concatenate(all_labels), np.concatenate(all_preds)

class TrainProgressCallback(pl.Callback):
    def __init__(self, split_label):
        super().__init__()
        self.split_label = split_label

    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        train_loss = metrics.get('train_loss')
        train_acc = metrics.get('train_acc')
        val_loss = metrics.get('val_loss')
        val_acc = metrics.get('val_acc')
        parts = [f"[{self.split_label}] epoch {trainer.current_epoch + 1}/{trainer.max_epochs}"]
        if train_loss is not None:
            parts.append(f"train_loss={float(train_loss):.4f}")
        if train_acc is not None:
            parts.append(f"train_acc={float(train_acc):.4f}")
        if val_loss is not None:
            parts.append(f"val_loss={float(val_loss):.4f}")
        if val_acc is not None:
            parts.append(f"val_acc={float(val_acc):.4f}")
        print(' | '.join(parts))

def train_model(config, split_label='train-val'):
    train_dl, val_dl = make_loaders(
        data_remapped['train']['patches'],
        data_remapped['train']['labels'],
        data_remapped['val']['patches'],
        data_remapped['val']['labels'],
        config['batch_size']
    )
    class_weights = build_class_weights(data_remapped['train']['labels'], NUM_CLASSES, config['class_weight_power'])
    model = SimpleCNN(in_channels=16, num_classes=NUM_CLASSES, dropout=config['dropout'])
    lightning_model = LandCoverClassifier(
        model,
        NUM_CLASSES,
        class_weights,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        loss_name=LOSS_NAME
    )

    print(f"Starting {split_label}: {config}")
    callbacks = [
        ModelCheckpoint(
            dirpath=OUTPUT_DIR / 'checkpoints',
            filename='cnn-manual-{epoch:02d}-{val_loss:.3f}',
            monitor='val_loss',
            mode='min',
            save_top_k=3
        ),
        EarlyStopping(monitor='val_loss', patience=PATIENCE, mode='min'),
        TrainProgressCallback(split_label)
    ]

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='cuda' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=callbacks,
        enable_checkpointing=True,
        log_every_n_steps=10,
        deterministic=True,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False
    )
    trainer.fit(lightning_model, train_dl, val_dl)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    y_true, y_pred = predict_with_model(lightning_model.model, val_dl, device)
    final_val_metrics = {
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)
    }
    print(f"Completed {split_label}: balanced_accuracy={final_val_metrics['balanced_accuracy']:.4f}, macro_f1={final_val_metrics['macro_f1']:.4f}")
    return lightning_model, trainer, final_val_metrics

print("\n" + "="*70)
print("MANUAL HYPERPARAMETER TUNING GUIDE")
print("="*70)
print("Try these next runs one at a time by copying a candidate into BEST_CONFIG:")
for candidate in TUNING_CANDIDATES:
    print(f"  - {candidate['name']}: {candidate}")

print("\n" + "="*70)
print("FINAL TRAINING")
print("="*70)
print(f"Config: {BEST_CONFIG}")
print(f"Max epochs: {MAX_EPOCHS}")
print(f"Early stopping: {PATIENCE} epochs")

lit_model, trainer, final_val_metrics = train_model(BEST_CONFIG, split_label='train-val')

print(f"Validation balanced accuracy with selected config: {final_val_metrics['balanced_accuracy']:.4f}")
print("\n✅ Training complete")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]



TRAINING
Max epochs: 50
Early stopping: 10 epochs
Device: <pytorch_lightning.accelerators.cuda.CUDAAccelerator object at 0x14b6e22cb0b0>



┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SimpleCNN │  384 K │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

Trainable params: 384 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 384 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

RuntimeError: weight tensor should be defined either for all 19 classes or no classes but got weight tensor of shape: [14]

## Evaluate on Test Set

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    model = model.to(device)
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Evaluating"):
            x = x.to(device)
            logits = model(x)
            preds = logits.argmax(1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.numpy())
    
    return np.concatenate(all_labels), np.concatenate(all_preds)

print("\n" + "="*70)
print("TEST SET EVALUATION")
print("="*70)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
y_true, y_pred = evaluate(lit_model.model, test_loader, device)

print(f"✓ Evaluated {len(y_true):,} samples")

## Compute Metrics

In [ ]:
# Overall metrics
overall_acc = (y_pred == y_true).mean()
balanced_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "="*70)
print("TEST METRICS")
print("="*70)
print(f"\nOverall Accuracy:  {overall_acc:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}  ← PRIMARY")
print(f"Macro-F1:          {macro_f1:.4f}  ← PRIMARY")

# Classification report
unique_classes = np.unique(np.concatenate([y_true, y_pred]))
target_names = [idx_to_label[i] for i in unique_classes]

report = classification_report(y_true, y_pred, labels=unique_classes,
                              target_names=target_names, digits=4, zero_division=0)

print("\n" + "="*70)
print("PER-CLASS REPORT")
print("="*70)
print(report)

# Comparison
print("\n" + "="*70)
print("COMPARISON")
print("="*70)
print(f"\nPrevious multiclass baseline: ~22-23% balanced accuracy")
print(f"Target for binary task:       ~70-85%")
print(f"Actual (this run):            {balanced_acc*100:.1f}%")

if balanced_acc > 0.70:
    print("\n🎉 Strong binary classification result!")
    print("   Cropland vs non-cropland is working well.")
else:
    print("\n⚠ Still below the target range for the binary task")
    print("   Try one of the manual tuning candidates next.")

## Visualizations

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=unique_classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=False, cmap='Blues',
            xticklabels=unique_classes, yticklabels=unique_classes,
            cbar_kws={'label': 'Normalized'}, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (Normalized)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'confusion_matrix.png', dpi=150)
plt.show()

print("✓ Confusion matrix saved")

In [ ]:
# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=unique_classes, zero_division=0
)

metrics_df = pd.DataFrame({
    'Class_ID': unique_classes,
    'Class_Name': [idx_to_label[i] for i in unique_classes],
    'Precision': precision,
    'Recall': recall,
    'F1': f1,
    'Support': support
}).sort_values('F1', ascending=False)

print("\nPer-class (sorted by F1):")
print(metrics_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metrics_df))
w = 0.25

ax.bar(x - w, metrics_df['Precision'], w, label='Precision', alpha=0.8)
ax.bar(x, metrics_df['Recall'], w, label='Recall', alpha=0.8)
ax.bar(x + w, metrics_df['F1'], w, label='F1', alpha=0.8)

ax.set_xlabel('Binary Class (sorted by F1)')
ax.set_ylabel('Score')
ax.set_title('Cropland vs Non-Cropland Performance')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Class_Name'], rotation=20)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figures' / 'per_class_metrics.png', dpi=150)
plt.show()

print("✓ Per-class plot saved")

## Save Results

In [ ]:
# Save metrics
results = {
    'overall_accuracy': float(overall_acc),
    'balanced_accuracy': float(balanced_acc),
    'macro_f1': float(macro_f1),
    'num_classes': int(NUM_CLASSES),
    'test_samples': int(len(y_true)),
    'selected_hyperparameters': BEST_CONFIG,
    'tuning_candidates': TUNING_CANDIDATES,
    'validation_balanced_accuracy': float(final_val_metrics['balanced_accuracy']),
    'validation_macro_f1': float(final_val_metrics['macro_f1']),
    'label_mapping': idx_to_label,
    'cropland_labels': sorted(cropland_labels),
    'research_question': 'Can a CNN classify cropland versus non-cropland in the Po Valley using multi-temporal Sentinel-2 imagery?',
    'classification_report': report,
    'normalization': 'GLOBAL_percentile_2_98',
    'improvement_note': 'Compare this binary result against the previous multiclass baseline (~22-23% balanced accuracy).'
}

with open(OUTPUT_DIR / 'results' / 'final_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

metrics_df.to_csv(OUTPUT_DIR / 'results' / 'per_class_metrics.csv', index=False)

print("\n" + "="*70)
print("✅ EVALUATION COMPLETE!")
print("="*70)
print(f"\nResults saved to: {OUTPUT_DIR / 'results'}")
print(f"Figures saved to: {OUTPUT_DIR / 'figures'}")
print(f"Model saved to: {OUTPUT_DIR / 'checkpoints'}")
print(f"\nFinal Balanced Accuracy: {balanced_acc*100:.1f}%")
print("Task: cropland vs non-cropland")